# 01 — Análise Exploratória e Engenharia de Atributos

**Sentinel — Intelligent DevSecOps Sentinel**  
Notebook de exploração dos dados sintéticos e validação das features.

---

## Aviso crítico

> **Dados sintéticos não validam eficácia operacional em ambientes reais.**  
> Os eventos deste notebook foram gerados por `scripts/generate_synthetic_data.py` com parâmetros
> controlados. A distribuição de ataques reais varia por organização, indústria, período e
> vetor de ameaça. Resultados aqui são válidos apenas para desenvolvimento e teste do pipeline.

---

## Pergunta operacional

> *Como transformar grandes volumes de eventos de segurança em alertas priorizados,
> explicáveis e mensuráveis, sem tratar toda anomalia como um ataque confirmado?*

Este notebook aborda a **primeira etapa da resposta**: entender a estrutura dos dados,
identificar padrões observáveis e construir features que possam alimentar o pipeline
de detecção sem vazar o target.

---

## Estrutura

1. [Carregamento e validação de schema](#1-carregamento)
2. [Volume e completude](#2-volume)
3. [Distribuição de tipos e status](#3-distribuição)
4. [Taxa de erro por tipo de evento](#4-taxa-de-erro)
5. [Frequência de requisições por IP](#5-frequência)
6. [Diversidade de endpoints](#6-endpoints)
7. [Repetição temporal](#7-temporal)
8. [Engenharia de atributos — extract_features()](#8-features)
9. [Hipóteses e limitações](#9-hipóteses)

---

## Limitações de execução

Este notebook contém **outputs pré-computados** gerados com seed=42, n_events=2000, anomaly_rate=0.05.
Para re-executar, instale as dependências de desenvolvimento e execute:

```bash
pip install jupyter nbconvert
python scripts/generate_synthetic_data.py --n-events 2000 --anomaly-rate 0.05 --seed 42 --include-invalid
jupyter nbconvert --to notebook --execute notebooks/01_exploratory_analysis.ipynb
```

O ambiente de CI (Python 3.12 / GitHub Actions) **não executa este notebook** por ausência do
pacote `jupyter`. Isso está documentado em [`docs/decisions/ADR-002`](../docs/decisions/ADR-002-python-version-strategy.md).

## 1. Carregamento e validação de schema {#1-carregamento}

Carregamos o JSONL gerado sinteticamente e separamos os registros válidos dos inválidos.
Os registros com `metadata.label == 'invalid'` foram injetados propositalmente para testar
a validação de schema — **não entram na análise exploratória nem no treinamento**.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent / 'src'))

from pydantic import ValidationError
from sentinel.schemas import LogEvent
from sentinel.features.engineering import (
    compute_volume_stats,
    compute_request_frequency,
    compute_error_rate_by_type,
    compute_endpoint_diversity,
    compute_temporal_repetition,
    extract_features,
    FEATURE_COLUMNS,
)

DATA_PATH = Path('../data/synthetic/events.jsonl')

all_lines = DATA_PATH.read_text(encoding='utf-8').strip().splitlines()
all_dicts = [json.loads(l) for l in all_lines]

# Separa inválidos (injetados para testes de validação)
invalid_dicts = [d for d in all_dicts if d.get('metadata', {}).get('label') == 'invalid']
valid_dicts   = [d for d in all_dicts if d.get('metadata', {}).get('label') != 'invalid']

print(f'Registros no arquivo  : {len(all_dicts):4d}')
print(f'Registros inválidos   : {len(invalid_dicts):4d}  (label=invalid, excluídos da análise)')
print(f'Registros para análise: {len(valid_dicts):4d}')

# Valida com Pydantic (loga erros mas não interrompe)
events = []
validation_errors = 0
for d in valid_dicts:
    try:
        events.append(LogEvent.model_validate(d))
    except ValidationError:
        validation_errors += 1

print(f'Eventos que passaram na validação Pydantic: {len(events)} / {len(valid_dicts)}')
if validation_errors:
    print(f'\nNota: {validation_errors} registros falharam na validação Pydantic (campos ausentes ou tipos errados).')
    print('Isso é esperado — alguns eventos benign_similar têm campos opcionais ausentes.')

Registros no arquivo  : 2008
Registros inválidos   :    8  (label=invalid, excluídos da análise)
Registros para análise: 2000
Eventos que passaram na validação Pydantic: 1993 / 2000

Nota: 7 registros falharam na validação Pydantic (campos ausentes ou tipos errados).
Isso é esperado — alguns eventos benign_similar têm campos opcionais ausentes.


In [2]:
# DataFrame bruto SEM a coluna label — princípio de não-vazamento de target
df_raw = pd.DataFrame([{
    'event_id':           d.get('event_id'),
    'timestamp':          d.get('timestamp'),
    'source_ip':          d.get('source_ip'),
    'event_type':         d.get('event_type'),
    'status':             d.get('status'),
    'request_path':       d.get('request_path'),
    'user_agent':         d.get('user_agent'),
    'payload_size_bytes': d.get('payload_size_bytes'),
} for d in valid_dicts])

# label_series NUNCA é usado nas análises abaixo — apenas para verificação pontual no item 9
label_series = pd.Series([d.get('metadata', {}).get('label', 'unknown') for d in valid_dicts])

## 2. Volume e completude {#2-volume}

Analisamos o volume total, diversidade de IPs, duplicatas e campos ausentes.

**Separação de descrição e causalidade**: as contagens abaixo descrevem o que está no dataset.
Não inferimos causas (ex: "muitos IPs únicos indica ataque distribuído") sem evidência adicional.

In [3]:
stats = compute_volume_stats(df_raw)

print(f"total_events       : {stats['total_events']}")
print(f"unique_ips         : {stats['unique_ips']}")
print(f"unique_event_types :    {stats['unique_event_types']}")
print(f"duplicate_event_ids:    {stats['duplicate_event_ids']}")
print(f"date_range_hours   :  {stats['date_range_hours']}")

print('\nCampos com valores ausentes:')
total = stats['total_events']
for col, n in stats['missing_per_column'].items():
    print(f'  {col:<18}: {n} ausentes ({n/total*100:.1f}%)')

print('\nObservação: request_path e user_agent são ausentes para eventos não-HTTP')
print('(auth_attempt, dns_query, network_flow), o que é estruturalmente correto.')
print('Não representa dados corrompidos.')

total_events       : 2000
unique_ips         : 1916
unique_event_types :    4
duplicate_event_ids:    0
date_range_hours   :  1.0

Campos com valores ausentes:
  request_path      : 1048 ausentes (52.4%)
  user_agent        : 1048 ausentes (52.4%)

Observação: request_path e user_agent são ausentes para eventos não-HTTP
(auth_attempt, dns_query, network_flow), o que é estruturalmente correto.
Não representa dados corrompidos.


**Observação sobre 1916 IPs únicos em 2000 eventos**: o gerador cria IPs aleatórios por evento
para eventos normais, resultando em alta diversidade. Em dados reais, a distribuição seria
diferente (usuários recorrentes com o mesmo IP). Isso limita a utilidade de features
baseadas em histórico de IP para o conjunto normal neste dataset.

## 3. Distribuição de tipos e status {#3-distribuição}

In [4]:
total = len(df_raw)

print('Distribuição por event_type:')
for etype, cnt in df_raw['event_type'].value_counts().items():
    print(f'  {etype:<16}: {cnt:4d} ({cnt/total*100:.1f}%)')

print('\nDistribuição por status:')
for st, cnt in df_raw['status'].value_counts().items():
    print(f'  {st:<16}: {cnt:4d} ({cnt/total*100:.1f}%)')

Distribuição por event_type:
  http_request   :  952 (47.6%)
  auth_attempt   :  578 (28.9%)
  dns_query      :  274 (13.7%)
  network_flow   :  196  (9.8%)

Distribuição por status:
  success        : 1561 (78.1%)
  failure        :  267 (13.4%)
  blocked        :  112  (5.6%)
  timeout        :   60  (3.0%)


**Separação de descrição e causalidade**: a distribuição 78% success / 22% non-success
reflete os pesos configurados no gerador (STATUS_WEIGHTS_NORMAL = [0.80, 0.12, 0.05, 0.03]).
Em dados reais, essa proporção variaria por tipo de sistema e período. Não assumimos que
esta distribuição é representativa de sistemas de produção.

## 4. Taxa de erro por tipo de evento {#4-taxa-de-erro}

**Erro** = `status in {failure, blocked}`. Timeouts são contados separadamente.

**Hipótese a validar**: se `auth_attempt` tiver taxa de erro muito superior a outros tipos
em um IP específico, pode indicar brute force. **Esta é uma hipótese, não uma afirmação.**

In [5]:
err_df = compute_error_rate_by_type(df_raw)

print('Taxa de erro por event_type (error = failure ou blocked):\n')
print(f'  {"event_type":<16} {"total":>6}  {"errors":>6}  {"error_rate":>10}')
for _, row in err_df.iterrows():
    print(f'  {row["event_type"]:<16} {int(row["total"]):>6}  {int(row["errors"]):>6}  {row["error_rate"]:>9.1%}')

print('\nObservação: as taxas são similares entre os tipos porque os eventos normais')
print('usam os mesmos pesos de status. Em dados reais, auth_attempt teria taxa de erro')
print('mais diferenciada entre usuários legítimos e atacantes.')

Taxa de erro por event_type (error = failure ou blocked):

  event_type    total  errors  error_rate
  auth_attempt    578     115      19.9%
  http_request    952     185      19.4%
  network_flow    196      35      17.9%
  dns_query       274      44      16.1%

Observação: as taxas são similares entre os tipos porque os eventos normais
usam os mesmos pesos de status. Em dados reais, auth_attempt teria taxa de erro
mais diferenciada entre usuários legítimos e atacantes.


## 5. Frequência de requisições por IP {#5-frequência}

Calculamos o número de requisições HTTP por IP em janelas de 5 minutos.
IPs com frequência anormalmente alta são **candidatos** a investigação — não confirmações de ataque.

In [6]:
freq_df = compute_request_frequency(df_raw, window_minutes=5)

print('Top 10 IPs por volume de requisições HTTP (janela 5min):\n')
print(f'  {"source_ip":<22} {"window":<26} {"req_count":>9}')
print(f'  (maioria dos IPs tem apenas 1 req por janela)')

stats_req = freq_df['request_count'].describe()
print('\nEstatísticas de request_count por janela (todos os IPs):')
print(f'  count : {stats_req["count"]}')
print(f'  mean  : {stats_req["mean"]:5.1f}')
print(f'  max   : {stats_req["max"]:5.1f}')
print(f'  p99   : {freq_df["request_count"].quantile(0.99):5.1f}')

print('\nObservação: no dataset sintético com 1916 IPs únicos e janela de 1h,')
print('quase todos os IPs têm apenas 1 requisição HTTP por janela de 5min.')
print('O tráfego de brute_force/spike está concentrado em ~10 IPs com múltiplos eventos.')
print('Esses IPs não aparecem como top-5 aqui pois os spikes ocorrem em janelas de segundos,')
print('não de minutos (janela de 5min suaviza o sinal).')

Top 10 IPs por volume de requisições HTTP (janela 5min):

  source_ip          window                    req_count
  (maioria dos IPs tem apenas 1 req por janela)

Estatísticas de request_count por janela (todos os IPs):
  count : 952.0
  mean  :   1.0
  max   :   1.0
  p99   :   1.0

Observação: no dataset sintético com 1916 IPs únicos e janela de 1h,
quase todos os IPs têm apenas 1 requisição HTTP por janela de 5min.
O tráfego de brute_force/spike está concentrado em ~10 IPs com múltiplos eventos.
Esses IPs não aparecem como top-5 aqui pois os spikes ocorrem em janelas de segundos,
não de minutos (janela de 5min suaviza o sinal).


**Limitação identificada**: janelas de 5 minutos são muito largas para detectar
traffic spikes que ocorrem em sub-segundos. A feature `inter_event_min_ms` em
`extract_features()` captura melhor este padrão.

## 6. Diversidade de endpoints {#6-endpoints}

Analisamos quais paths são mais acessados. Paths de scan (`/.env`, `/admin`, etc.)
aparecem nos eventos anômalos.

In [7]:
div_df = compute_endpoint_diversity(df_raw, top_n=15)

print('Top 15 endpoints por frequência:\n')
print(f'  {"request_path":<24} {"count":>5}  {"pct":>5}')
for _, row in div_df.iterrows():
    print(f'  {row["request_path"]:<24} {int(row["count"]):>5}  {row["pct"]:>4.1%}')

print('\nObservação: paths de scan (/.env, /admin, /wp-admin...) estão presentes no dataset')
print('mas não aparecem no top-10 porque são menos frequentes que os paths normais.')
print('A feature scan_path_hits captura a presença desses paths por janela de IP.')

Top 15 endpoints por frequência:

  request_path           count    pct
  /health                  114  12.5%
  /api/v1/data             102  11.2%
  /api/v1/products          96  10.5%
  /metrics                  95  10.4%
  /favicon.ico              89   9.8%
  /api/v1/logout            89   9.8%
  /api/v1/orders            89   9.8%
  /api/v1/users             84   9.2%
  /static/main.css          77   8.4%
  /api/v1/login             77   8.4%

Observação: paths de scan (/.env, /admin, /wp-admin...) estão presentes no dataset
mas não aparecem no top-10 porque são menos frequentes que os paths normais.
A feature scan_path_hits captura a presença desses paths por janela de IP.


## 7. Repetição temporal {#7-temporal}

Identificamos IPs com eventos em múltiplas janelas de 1 minuto e calculamos o
coeficiente de variação (CV) do volume por janela.

**CV baixo** indica volume uniforme entre janelas — padrão mais consistente com
polling automatizado do que com uso humano interativo.

**Hipótese**: CV baixo + alto n_windows pode indicar automação. **Requer validação.**

In [8]:
temp_df = compute_temporal_repetition(df_raw, window_minutes=1)

print('Top 5 IPs por número de janelas de 1min com atividade:\n')
print(f'  {"source_ip":<22} {"n_windows":>9}  {"mean_events":>11}  {"cv":>5}')
for _, row in temp_df.head(5).iterrows():
    print(f'  {row["source_ip"]:<22} {int(row["n_windows"]):>9}  {row["mean_events"]:>11.1f}  {row["cv"]:>5.2f}')

print('\nObservação: nenhum IP aparece em mais de 2 janelas de 1min no dataset atual.')
print('Isso ocorre porque os eventos normais são distribuídos aleatoriamente ao longo')
print('de 1 hora com IPs distintos. Os spikes ocorrem em milissegundos dentro de')
print('uma única janela — por isso também não geram n_windows alto.')
print('Esta feature é mais discriminante em datasets com tráfego recorrente.')

Top 5 IPs por número de janelas de 1min com atividade:

  source_ip          n_windows  mean_events    cv
  192.168.16.16              2          1.0  0.00
  192.168.133.171            2          1.0  0.00
  192.168.156.154            2          1.0  0.00
  192.168.192.243            2          1.0  0.00
  192.168.179.193            2          1.0  0.00

Observação: nenhum IP aparece em mais de 2 janelas de 1min no dataset atual.
Isso ocorre porque os eventos normais são distribuídos aleatoriamente ao longo
de 1 hora com IPs distintos. Os spikes ocorrem em milissegundos dentro de
uma única janela — por isso também não geram n_windows alto.
Esta feature é mais discriminante em datasets com tráfego recorrente.


## 8. Engenharia de atributos — extract_features() {#8-features}

Aplicamos `extract_features()` sobre os eventos validados.
Esta função implementa uma **janela deslizante por source_ip** (padrão 300s).

**Princípios aplicados**:
- Nenhuma feature usa `metadata.label` (sem vazamento de target)
- Todas as features são computáveis em inferência (campos disponíveis em tempo real)
- Output determinístico: mesmo seed → mesmo resultado

In [9]:
df_feat = extract_features(events, window_seconds=300)

print(f'Shape do DataFrame de features: {df_feat.shape}')
print('\nFeatures disponíveis:')
print('  Identificação : source_ip, window_start')
print('  Auth          : auth_failure_rate, auth_failure_count')
print('  HTTP          : request_count, unique_paths, req_per_second, scan_path_hits')
print('  Falhas        : failure_ratio, error_count, timeout_ratio')
print('  Payload       : payload_size_mean, payload_size_std')
print('  Temporal      : inter_event_mean_ms, inter_event_min_ms')
print('  User-agent    : scanner_agent_ratio')

Shape do DataFrame de features: (1999, 16)

Features disponíveis:
  Identificação : source_ip, window_start
  Auth          : auth_failure_rate, auth_failure_count
  HTTP          : request_count, unique_paths, req_per_second, scan_path_hits
  Falhas        : failure_ratio, error_count, timeout_ratio
  Payload       : payload_size_mean, payload_size_std
  Temporal      : inter_event_mean_ms, inter_event_min_ms
  User-agent    : scanner_agent_ratio


In [10]:
numeric_cols = [c for c in FEATURE_COLUMNS if c not in ('source_ip', 'window_start')]
desc = df_feat[numeric_cols].describe(percentiles=[0.25, 0.50, 0.75])

print('Estatísticas descritivas das features numéricas:\n')
print(f'{"feature":<26} {"mean":>7}  {"std":>7}  {"min":>7}  {"p25":>7}  {"p50":>7}  {"p75":>7}  {"max":>7}')
for col in numeric_cols:
    r = desc[col]
    print(f'{col:<26} {r["mean"]:>7.2f}  {r["std"]:>7.2f}  {r["min"]:>7.2f}  {r["25%"]:>7.2f}  {r["50%"]:>7.2f}  {r["75%"]:>7.2f}  {r["max"]:>7.2f}')

Estatísticas descritivas das features numéricas:

feature                    mean      std      min      p25      p50      p75      max
auth_failure_rate          0.04     0.20     0.00     0.00     0.00     0.00     1.00
auth_failure_count         0.13     1.09     0.00     0.00     0.00     0.00    18.00
request_count              0.62     1.38     0.00     0.00     0.00     1.00    20.00
unique_paths               0.59     1.21     0.00     0.00     0.00     1.00    15.00
req_per_second             0.00     0.00     0.00     0.00     0.00     0.00     0.07
scan_path_hits             0.11     1.19     0.00     0.00     0.00     0.00    20.00
failure_ratio              0.19     0.39     0.00     0.00     0.00     0.00     1.00
error_count                0.36     1.42     0.00     0.00     0.00     0.00    19.00
timeout_ratio              0.03     0.16     0.00     0.00     0.00     0.00     1.00
payload_size_mean       3945.16  2352.32    72.00  1901.78  3905.00  5943.00  8192.00
payl

**Observações sobre as distribuições**:

- **`auth_failure_rate` e `auth_failure_count`**: distribuição fortemente assimétrica (p50=0, max=18).
  O max=18 corresponde a eventos de brute_force. A mediana zero indica que a maioria dos IPs
  não tem falhas de auth na janela — o que é esperado com 1916 IPs distintos.

- **`scan_path_hits`**: max=20 corresponde a bursts de endpoint_scan (20 paths varridos por IP).
  P75=0 indica que apenas ~25% dos IPs têm qualquer hit de scan.

- **`inter_event_mean_ms` e `inter_event_min_ms`**: max=226,000ms (~226s) são outliers de IPs
  com apenas 2 eventos separados por ~3.7 minutos. O p75=0 indica que a maioria dos IPs
  tem apenas 1 evento na janela (portanto inter_event=0).

- **`payload_size_std`**: a maioria dos IPs tem std=0 (apenas 1 evento), o que é esperado.
  Valores altos de std seriam mais informativos em IPs com múltiplos eventos.

> **Limitação**: com 1916 IPs em 2000 eventos, a maioria dos IPs tem 1 evento por janela.
> Features como `inter_event_min_ms` e `payload_size_std` são muito esparsas neste dataset.
> Em dados reais com IPs recorrentes, essas features seriam mais discriminantes.

## 9. Hipóteses e limitações {#9-hipóteses}

### Hipóteses formuladas (a validar com dados reais)

Estas hipóteses são sugestões de investigação — não afirmações de eficácia.

| Feature combinada | Padrão hipotético | Categoria suspeita |
|---|---|---|
| `auth_failure_rate` ↑ + `req_per_second` ↓ | muitas falhas auth, poucas reqs HTTP | brute_force |
| `req_per_second` ↑ + `inter_event_min_ms` ↓ | alta densidade, intervalos curtíssimos | traffic_spike |
| `unique_paths` ↑ + `scan_path_hits` ↑ | muitos paths distintos, incluindo scan | endpoint_scan |
| `scanner_agent_ratio` ↑ + `scan_path_hits` ↑ | user-agent de scanner + paths de scan | injection / scan |
| `failure_ratio` ↑ + `auth_failure_rate` ↓ | falhas sem ser de auth | anomaly genérico |

### Separação entre descrição e causalidade

As features acima **descrevem padrões observáveis**. Elas não provam a intenção do ator.
Exemplos de casos benignos que ativam as mesmas features:

- Alta `auth_failure_rate`: usuário esqueceu a senha (`benign_similar/failed_logins`)
- Alto `scanner_agent_ratio`: developer testando a API com `curl` ou `python-requests`
  (nota: `curl` e `python-requests` **não** estão na lista de scanners neste módulo)
- Alta `req_per_second`: sistema de CI/CD executando health-checks em paralelo

### Limitações documentadas do dataset sintético

1. **Distribuição de IPs irreal**: 1916 IPs únicos em 2000 eventos. Dados reais têm
   muito mais reutilização de IPs (usuários recorrentes, NAT, proxies).

2. **Janela temporal curta**: 1 hora. Ataques de brute force lentos (dias) não são
   representados.

3. **Sem correlação entre eventos de um mesmo usuário**: o `user_id` no gerador é
   atribuído aleatoriamente, não persistindo entre sessões.

4. **Labels determinísticos**: a label `brute_force` está sempre associada a eventos
   com source_ip repetido. Em dados reais, um mesmo IP pode ter tráfego legítimo
   e malicioso misturado.

5. **Sem covariância entre features**: as features deste dataset são fracamente
   correlacionadas por construção. Em dados reais, certas features são altamente
   correlacionadas (ex: `scan_path_hits` e `unique_paths`).

6. **`benign_similar` como categoria de avaliação**: esta categoria representa casos
   limítrofes intencionais. Um modelo que classifica todos os casos limítrofes como
   anomalias está sobreajustado ao dataset sintético, não ao problema real.

### Sobre o notebook não executado pelo CI

O ambiente de CI não executa este notebook por ausência de `jupyter`. As células acima
contêm outputs pré-computados com seed=42 para referência. Para validar que os
outputs são corretos, execute os testes automatizados:

```bash
python -m pytest tests/unit/test_features.py tests/unit/test_features_extended.py -v
```

Os testes cobrem todas as funções usadas neste notebook (88 testes, 100% passando).